# 02 · Проверка модели перед экспериментами

Один раз на каждую новую модель или новый chat template. Ничего не пишет в журнал,
кроме заметки в конце (по желанию).

Проверяем:
1. какой Python и устройство (MPS на Mac);
2. **окна позиций глазами** — что `field_read`, `post_message`, `pre_action`
   стоят там, где должны (см. AGENTS.md: на Qwen3.x арифметика окон уже ломалась);
3. формат вызова инструмента, выведенный из шаблона;
4. что хук steering меняет ровно столько позиций, сколько в окне;
5. сколько времени займёт полный прогон.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz
from steering_tool_feedback import doctor, simulator as S, steering as ST
from steering_tool_feedback.windows import describe_windows
import numpy as np
doctor.main([])

python      3.12.14  /Users/admin/Desktop/steering-tool-feedback/.venv/bin/python
prefix      /Users/admin/Desktop/steering-tool-feedback/.venv
venv        yes
user site   /Users/admin/.local/lib/python3.12/site-packages  enabled=False

project     installed 0.1.0  /Users/admin/Desktop/steering-tool-feedback/src/steering_tool_feedback/__init__.py
commands    stf-* NOT on PATH
            installed in: /Users/admin/Desktop/steering-tool-feedback/.venv/bin
            -> export PATH="/Users/admin/Desktop/steering-tool-feedback/.venv/bin:$PATH"   (добавить в ~/.zshrc)
            -> или без PATH: python3 -m steering_tool_feedback doctor|windows|run

[core]
  numpy          2.5.4        /Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/numpy/__init__.py
  scipy          1.18.1       /Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/scipy/__init__.py
  sklearn        1.9.1        /Users/admin/Desktop/steering-tool-feedback/.venv/lib/

  torch          2.14.1       /Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/torch/__init__.py


/Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  transformers   5.20.1       /Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/transformers/__init__.py
  accelerate     1.15.0       /Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/accelerate/__init__.py

[dev]


  pytest         9.1.1        /Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/pytest/__init__.py

[optional]


W1019 20:44:42.321000 4931 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


  peft           0.21.2       /Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/peft/__init__.py


  trl            1.16.0       /Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/trl/__init__.py

torch device  mps

core deps OK. Для моделей: pip install -e '.[model]'; для тестов: '.[dev]'


0

In [2]:
MODEL = "Qwen/Qwen3-1.7B"     # HF id или локальная папка; меняй здесь
cfg = ExperimentConfig.make(name="model-check", hypothesis="explore",
                            notebook="notebooks/02_model_check.ipynb",
                            overrides={"model.name": MODEL})
exp = Experiment(cfg)
exp.data()
be = exp.backend
be.info()

[20:44:43] run 20261019-204443_model-check_0e49d9  ->  results/runs/20261019-204443_model-check_0e49d9
[20:44:43] data: {'train': 59, 'val': 24, 'test': 37}  types={'invariant': 37, 'stress': 11, 'changing': 72}  positions={'early': 95, 'late': 25}
[20:44:43] loading model Qwen/Qwen3-1.7B …


Could not reach the Hub ([SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)). Using cached commit hash for 'Qwen/Qwen3-1.7B'.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   1%|          | 2/311 [00:00<00:40,  7.61it/s]

Loading weights:  90%|████████▉ | 279/311 [00:00<00:00, 978.71it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 813.68it/s]

{'backend': 'hf',
 'model': 'Qwen/Qwen3-1.7B',
 'device': 'mps',
 'dtype': 'float16',
 'n_layers': 28,
 'hidden_size': 2048,
 'chat_template_hash': 'a55ee1b1660128b7',
 'template_kwargs': {'enable_thinking': False},
 'action_format': None}

### Окна позиций — смотреть глазами

In [3]:
t = next(x for x in exp.state["splits"]["train"] if x.pair_type == "changing")
for b in (0, 1):
    p = exp._prompt(t, b)
    print(f"--- {t.task_id} branch {b}: allowed {sorted(t.branch(b).allowed_actions)}")
    if p.prepared is not None:
        print(describe_windows(be.tok, p.prepared))
print("action format:", be.info()["action_format"])

--- t0066 branch 0: allowed ['proceed']
sequence: 763 tokens
window tool_content  [722, 753)  31 tok
  … '\n' '<tool_response>' '\n' ⟦ '{"' 'doc' '_id' '":' ' "' 'DOC' '-' '9' '8' '1' '",' ' "' 'access' '":' ' "' 'gr' 'anted' '",' ' "' 'content' '":' ' "' 'Q' '3' ' revenue' ' rose' ' ' '4' '%' '."' '}\n' ⟧ '</tool_response>' '<|im_end|>' '\n' …
  text: '{"doc_id": "DOC-981", "access": "granted", "content": "Q3 revenue rose 4%."}\n'
window field_read    [734, 740)  6 tok
  … '1' '",' ' "' ⟦ 'access' '":' ' "' 'gr' 'anted' '",' ⟧ ' "' 'content' '":' …
  text: 'access": "granted",'
window post_message  [749, 756)  7 tok
  … ' revenue' ' rose' ' ' ⟦ '4' '%' '."' '}\n' '</tool_response>' '<|im_end|>' '\n' ⟧ '<|im_start|>' 'assistant' '\n' …
  text: '4%."}\n</tool_response><|im_end|>\n'
window pre_action    [756, 763)  7 tok
  … '</tool_response>' '<|im_end|>' '\n' ⟦ '<|im_start|>' 'assistant' '\n' '<think>' '\n\n' '</think>' '\n\n' ⟧  …
  text: '<|im_start|>assistant\n<think>\n\n</think>\n\

### Хук срабатывает ровно в окне на этой модели

In [4]:
from steering_tool_feedback.backends import ResolvedSteer
p = exp._prompt(t, 1)
vec = ST.SteeringVector(be.n_layers // 2, np.ones(be.hidden_size, dtype=np.float32), "post_message", {})
for kind in ("field_read", "post_message", "pre_action"):
    w = p.windows[kind]
    scores, hook = be.score_actions(p.prepared.input_ids, ResolvedSteer(vec, w, 1.0))
    print(f"{kind:13} window={len(w):3} applied={hook.n_applied:3} positions from {hook.positions_source}")

field_read    window=  6 applied=  6 positions from position_ids


post_message  window=  7 applied=  7 positions from position_ids


pre_action    window=  7 applied=  7 positions from position_ids


### Что модель выбирает без вмешательства и сколько это стоит по времени

In [5]:
for b in (0, 1):
    d = be.decide(exp._prompt(t, b), "scored", None, 0.0, 0)
    top = sorted(d.scores.items(), key=lambda kv: -kv[1])[:3]
    print(f"branch {b}: {d.action:20} ok={d.action in t.branch(b).allowed_actions}  "
          f"{d.latency_ms:.0f} ms  top3={[(a, round(s, 2)) for a, s in top]}")
exp.estimate()

branch 0: proceed              ok=True  1443 ms  top3=[('proceed', -0.0), ('ask_user', -14.84), ('stop_no_option', -22.25)]


branch 1: request_access       ok=True  397 ms  top3=[('request_access', 0.0), ('ask_user', -25.72), ('retry_call', -26.45)]


[20:44:58] estimate: ~38 мин всего (0.40 с на решение, 1.96 с на шаг эпизода, 1.02 с на съём активаций); повторные запуски берут готовое из кэша


,stage,calls,sec_per_call,minutes
0,baseline,166,0.401,1.1
1,probe,118,1.016,2.0
2,sweep,2160,0.401,14.4
3,fit,144,0.401,1.0
4,test,518,0.401,3.5
5,episodes,495,1.961,16.2


In [6]:
# По желанию — зафиксировать в журнале, что модель проверена:
# journal.note(f"Модель {MODEL} проверена: окна ок, формат {be.info()['action_format']}, "
#              f"оценка полного прогона ~N мин на {be.info()['device']}.", tag="модель")